# Acervo que Fala — Notebook 04 (v5): duas correções cirúrgicas e congelamento

**Projeto final** · Inteligência Artificial Generativa & Large Language Models (ICA/PUC-Rio) · Eduardo Tosto

Este notebook fecha o pipeline: observação → redação estruturada (alt-text + descrição do objeto + flags) num lote de **20 objetos** (5 do smoke test + 15 novos, fora do conjunto de avaliação).

**O que mudou da v4 para a v5:** o lote v4 mostrou que o modelo atingiu o **teto de obediência a prompt** — com 25 regras concorrendo, duas produziram efeito colateral: a regra dos pares cor↔ave fez o modelo **inventar** "penas de arara" num objeto cujo registro não nomeia ave nenhuma; e a regra de "uma atribuição por texto" deixou fatos do catálogo (a roseta do Abano, que ESTÁ no registro) sem marca, lendo como se fossem visíveis na foto. O **prompt v8** aplica só duas correções cirúrgicas — (1) ave citada SOMENTE com fonte no registro; (2) atribuição por FATO, não por texto — mais três apertos de regressão ("adquirido em", povo sempre no alt, fundo de estúdio não é flag). Depois desta rodada o prompt **congela** e o projeto segue para a avaliação (E8–E10). O resultado desta v5 também alimenta o **bake-off de redator** (Notebook 05, Gemma 3).

*Metodologia: projeto construído por um designer com LLMs como suporte (vibe coding) — cada célula explicada.*

### Como rodar
1. **Ambiente de execução → Alterar o tipo → GPU T4** · 2. **Executar tudo** · 3. Tempo: **~35–45 min**. Deixe a aba aberta durante a execução.

In [ ]:
# Etapa 1 — Instalação (Pillow travada, regra da casa) + checagem do ambiente
import PIL
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers pillow=={PIL.__version__}
import torch, transformers
from PIL import ImageDraw
from torchvision.io import decode_image
print(f"transformers {transformers.__version__} | GPU: {torch.cuda.is_available()}")
print("ambiente íntegro ✓")

## Etapa 2 — Buscar os 20 objetos, com salvaguardas de imagem (~3 min)

O lote: os **5 objetos do smoke test** (para comparar com os notebooks anteriores) + **15 novos**, sorteados com seed fixa (reproduzível) entre os itens que **não** estão no conjunto de avaliação — o lote serve para testar o pipeline em escala, sem "viciar" nos itens que depois vão dar a nota.

Duas salvaguardas novas ao baixar cada foto:

- **Orientação EXIF**: fotos de câmera guardam a rotação numa etiqueta interna que os navegadores aplicam, mas o Python não — sem esta linha, o modelo poderia receber uma foto deitada sem ninguém saber. `ImageOps.exif_transpose` aplica a rotação correta.
- **Conversão para RGB**: garante que qualquer foto (escala de cinza, outros formatos de cor) chegue ao modelo no formato esperado.

Desta vez o registro completo de cada objeto (povo, materiais, dimensões, descrição curatorial...) **viaja junto** — a correção do bug do Notebook 03.

In [ ]:
import io, re, requests
from PIL import Image, ImageOps

BASE = "https://tainacan.museudoindio.gov.br/wp-json/tainacan/v2"
IDS_SMOKE = [9196, 665, 51023, 63283, 78838]
# 15 novos: sorteio seed 42, estratificado por categoria, excluindo os 50 casos
# de avaliação (seleção documentada no repositório, commit da E7)
IDS_LOTE = [1376, 84811, 883523, 2081, 5011, 200648, 210680, 5146, 500179, 3411, 1366, 4156, 205095, 905, 500322]

CAMPOS_REGISTRO = ["Nome do item", "Povo", "Categoria", "Matéria-prima",
                   "Técnica de confecção", "Dimensões", "Função",
                   "Estado de origem", "Ano de aquisição do objeto", "Descrição"]

objetos = []
for item_id in IDS_SMOKE + IDS_LOTE:
    item = requests.get(f"{BASE}/items/{item_id}", timeout=60).json()
    url_imagem = re.search(r'src="([^"]+)"', item["document_as_html"]).group(1)
    foto = Image.open(io.BytesIO(requests.get(url_imagem, timeout=90).content))
    foto = ImageOps.exif_transpose(foto).convert("RGB")  # salvaguardas
    meta_bruto = requests.get(f"{BASE}/item/{item_id}/metadata", timeout=60).json()
    todos = {m["metadatum"]["name"]: m["value_as_string"] for m in meta_bruto if m.get("value_as_string")}
    registro = {c: todos.get(c, "") for c in CAMPOS_REGISTRO}
    objetos.append({"id": item_id, "titulo": item["title"], "foto": foto, "registro": registro})
    print(f"✓ {item_id} — {item['title']} ({registro['Povo']})")
print(f"{len(objetos)} objetos carregados")

In [ ]:
# Etapa 3 — Drive (rubrica v1.1) + embeddings do RAG (como no Notebook 03)
import json, os
from google.colab import drive
from sentence_transformers import SentenceTransformer, util

drive.mount("/content/drive")
PROJETO = "/content/drive/MyDrive/00_IA/GenAI & LLMs - PUC/Projeto_LLM"
# v1.1: rubrica atualizada com as regras da revisão editorial (arquivo novo,
# versionado — a v1.0 continua no Drive como rubrica.json)
with open(f"{PROJETO}/dados/rubrica_v1_1.json", encoding="utf-8") as f:
    rubrica = json.load(f)
trechos = rubrica["trechos"]

embedder = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
vetores = embedder.encode([t["texto"] for t in trechos], convert_to_tensor=True)

def recuperar(consulta, k=3):
    v = embedder.encode(consulta, convert_to_tensor=True)
    scores = util.cos_sim(v, vetores)[0]
    achados = []
    for i in scores.argsort(descending=True).tolist():
        if trechos[i]["categoria"] == "geral":
            continue
        achados.append(trechos[i])
        if len(achados) == k:
            break
    return achados

print(f"rubrica {rubrica['versao']}: {len(trechos)} trechos indexados ✓")

In [ ]:
# Etapa 4 — Modelo (Qwen3-VL-8B em 4-bit, como nos notebooks anteriores)
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODELO = "Qwen/Qwen3-VL-8B-Instruct"
quantizacao = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
modelo = Qwen3VLForConditionalGeneration.from_pretrained(
    MODELO, quantization_config=quantizacao, device_map="auto"
)
processador = AutoProcessor.from_pretrained(MODELO)

def gerar(conteudo, max_tokens=400):
    conversa = [{"role": "user", "content": conteudo}]
    entradas = processador.apply_chat_template(
        conversa, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(modelo.device)
    with torch.no_grad():
        saida = modelo.generate(**entradas, max_new_tokens=max_tokens)
    return processador.decode(saida[0][entradas["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def extrair_json(texto):
    texto = re.sub(r"^```(json)?|```$", "", texto.strip(), flags=re.MULTILINE).strip()
    inicio, fim = texto.find("{"), texto.rfind("}")
    return json.loads(texto[inicio:fim + 1])

print("modelo carregado ✓")

## Etapa 5 — Observação visual dos 20 (~15 min)

Mesmo prompt v2 dos notebooks anteriores (estável desde a E5). A célula imprime o progresso a cada objeto.

In [ ]:
PROMPT_OBSERVACAO_V2 = (
    "Descreva APENAS o que está visível nesta fotografia de um objeto de museu:\n"
    "- formas, cores e materiais aparentes — incluindo cores de bordas, faixas e acabamentos, não só as dominantes;\n"
    "- a posição/orientação do objeto (de pé, inclinado, deitado) e se partes internas (boca, interior, verso) estão visíveis;\n"
    "- o enquadramento: o objeto aparece inteiro ou só um detalhe/close?;\n"
    "- o fundo e qualquer artefato de estúdio (etiqueta, numeração, cartela de cores, régua, suporte).\n"
    "NÃO invente o que não dá para ver. Se algo estiver ilegível ou incerto, diga isso em vez de estimar. "
    "Responda em português."
)

for n, obj in enumerate(objetos, 1):
    obj["observacao"] = gerar(
        [{"type": "image", "image": obj["foto"]}, {"type": "text", "text": PROMPT_OBSERVACAO_V2}]
    )
    print(f"[{n}/{len(objetos)}] {obj['titulo']} observado ✓")

## Etapa 6 — Redação estruturada v8: correções cirúrgicas, não regras novas (~20 min)

O prompt v8 mantém as 25 regras editoriais e muda só o que o lote v4 provou estar quebrado:

- **Ave só com fonte**: par cor↔ave SOMENTE quando a matéria-prima ou a descrição do registro nomeia a ave; registro sem ave → só as cores, nenhuma espécie inventada. (Na v4, a regra do par empurrou o modelo a inventar "penas de arara" onde não havia fonte.)
- **Atribuição por fato, não por texto**: TODO fato que vem do catálogo e não é visível na foto carrega marca de atribuição, com formulação variada. (Na v4, uma marca única no início deixou a roseta do Abano — que está no registro — lendo como observação visual.)
- Apertos de regressão: "adquirido em" (nunca "foi aquisição em"), povo sempre no alt, e **fundo de estúdio não é artefato** — flag só para etiqueta/numeração/cartela/régua/suporte, e flag nunca afirma ausência (na v4, 12 das 18 flags eram ruído de fundo).

O modelo continua sem ver a imagem nesta etapa — recebe a observação, o registro completo e as diretrizes recuperadas da rubrica v1.1.

In [ ]:
PROMPT_REDACAO_V8 = """Você escreve descrições de acessibilidade para o acervo digital de um museu, lidas por pessoas cegas via leitor de tela. Escreva em linguagem cotidiana — NUNCA jargão de catálogo ('globular' → 'arredondado'), NUNCA palavras inventadas; termo técnico só se vier do registro ou do glossário. Só afirmações verificáveis: proibido 'sugere', 'parece', 'parecendo X ou Y', 'possivelmente'; proibido inferir uso ou desgaste ('sinais de uso', 'uso frequente'); proibido juízo estético ('composição rica') e frases vazias ('porte médio', 'forma funcional', 'comprimento para uso prático'). Não repita a mesma palavra-chave várias vezes.

OBSERVAÇÃO VISUAL DA FOTOGRAFIA (única fonte do que é visível):
{observacao}

REGISTRO DO MUSEU (fatos do catálogo — o título nomeia o objeto):
{registro}

DIRETRIZES PARA ESTE TIPO DE OBJETO (recuperadas da base do projeto):
{diretrizes}

PRODUZA TRÊS SAÍDAS:

A) alt_text — uma frase, máx. 30 palavras, descrevendo a FOTOGRAFIA. Começa pelo objeto (nomeado pelo TÍTULO do registro — nunca rebatize pela aparência) e pelo povo — o nome do povo SEMPRE aparece no alt_text. Enquadramento: marque 'Detalhe de...' SÓ quando a foto mostra claramente um fragmento; objeto que encosta ou sangra nas margens conta como completo; NUNCA escreva 'inteiro', 'horizontal' ou 'vertical' — não informam. O fundo do estúdio NUNCA aparece nem empresta cor ao objeto; quando a cor de base é da própria peça, nomeie pelo material ('sobre a argila bege'), nunca como 'fundo'. Cores: nomeie onde informam (penas, miçangas, pinturas — proibido 'colorido', 'tons variados'); em material natural sem tingimento, nomeie o MATERIAL em vez da cor. Padrões têm FORMA além de cor (faixas, xadrez, losangos); padrão abstrato é 'geométrico' — nunca vire flores ou corações. Conte partes distinguíveis (ex.: 'sete tubos'). Penas: par cor+ave SOMENTE quando a Matéria-prima ou a Descrição do registro NOMEIA a ave ('penas vermelhas e amarelas de arara'); se o registro não nomeia nenhuma ave, descreva só as cores — citar espécie sem fonte é o pior erro possível. Na dúvida de material: termo genérico OU a matéria-prima do registro — nunca 'parecendo cerâmica ou madeira'. ARTEFATO DE ESTÚDIO/INVENTÁRIO NUNCA APARECE (vale também para a saída B) — ERRADO: '...com marcação numérica na base.' CERTO: terminar a frase sem citar a marcação (ela vira flag).

B) descricao_objeto — descreve o OBJETO, não a fotografia: PROIBIDO posição, inclinação, fundo, enquadramento ou a foto; relação que depende do ponto de vista ('mais curtos no topo') vira relação da própria peça ('em cascata', 'decrescentes'). PROIBIDO citar artefato de estúdio/inventário — ERRADO: 'A base apresenta uma marcação numérica, que não é parte do objeto original.' CERTO: a frase termina sem citar a marcação (ela já virou flag). Em amostras e conteúdos (argila, resina, sementes), o CONTEÚDO vem antes do recipiente. Dois parágrafos:
   1º: abre direto com o objeto e sua função — mas só função que ACRESCENTA (caça, ritual, preparo de alimentos); nunca o óbvio ('pulseira usada no pulso'). Ex.: 'Um pote cerâmico Karajá que, segundo o registro do museu, era usado no preparo e serviço de alimentos.' PROIBIDO abrir com 'O objeto é...', 'Trata-se de...' ou anunciar 'A função é...'. Depois, a aparência: formas, materiais e padrões em palavras comuns, informações do mesmo gênero agrupadas (material dito UMA vez, num lugar só).
   2º: os demais fatos do catálogo. Escala: a MAIOR dimensão aproximada ('cerca de 40 cm de comprimento'); miniatura é declarada ('miniatura de 6 cm, cabe na palma da mão'); escala óbvia (pulseira) pode ser omitida. Aves das penas: detalhe cor a cor conforme o registro. Significado cultural: só se estiver no registro.
   REGRA DE ATRIBUIÇÃO (vale para o texto todo): TODO fato que vem do catálogo e NÃO é visível na foto — função, técnica, origem, ano, medidas, decorações que o registro descreve — carrega uma marca de atribuição na própria frase, com formulação variada: 'segundo o registro do museu', 'o registro informa', 'conforme o catálogo', 'de acordo com o registro'. O que a observação viu não leva marca. Sem a marca, quem ouve não distingue o que a foto mostra do que o museu documentou. Datas sempre em frase natural: 'adquirido em 1977' — NUNCA 'foi aquisição em'.
   NUNCA afirme ausências ('sem etiquetas', 'não há sinais de...') — o que não existe simplesmente não aparece no texto.

C) flags — lista do que precisa de revisão humana:
   - tipo 'artefato_estudio': etiqueta, numeração, cartela, régua ou suporte que a observação notou — cada um vira uma flag. O FUNDO LISO DE ESTÚDIO NÃO É ARTEFATO e não vira flag. Uma flag descreve o que EXISTE — nunca escreva flag afirmando ausência ('sem etiquetas visíveis');
   - tipo 'divergencia_imagem_catalogo': algo claramente visível que o registro não menciona, ou objeto visto diferente do que o título nomeia (nesse caso, use o título no texto e registre a divergência aqui);
   - tipo 'metadado_suspeito': valor do registro que parece improvável (dimensão absurda para o tipo de objeto, data impossível) ou contradição entre campos do registro (ex.: Descrição diz 'brinquedo em miniatura' e Função diz 'utilizado para caça').
   Lista vazia [] se não houver nada.

Responda APENAS com JSON: {{"alt_text": "...", "descricao_objeto": "...", "flags": [{{"tipo": "...", "detalhe": "..."}}]}}"""

for n, obj in enumerate(objetos, 1):
    registro_txt = "\n".join(f"{k}: {v}" for k, v in obj["registro"].items() if v)
    consulta = f"{obj['titulo']} ({obj['registro']['Categoria']}). {obj['observacao'][:250]}"
    achados = recuperar(consulta)
    obj["diretrizes_usadas"] = [t["id"] for t in achados]
    prompt = PROMPT_REDACAO_V8.format(
        observacao=obj["observacao"],
        registro=registro_txt,
        diretrizes="\n".join(f"- {t['texto']}" for t in achados),
    )
    resposta = gerar([{"type": "text", "text": prompt}], max_tokens=700)
    try:
        saida = extrair_json(resposta)
        obj["alt_text"] = saida["alt_text"]
        obj["descricao_objeto"] = saida["descricao_objeto"]
        obj["flags"] = saida.get("flags", [])
        obj["json_valido"] = True
    except Exception:
        obj["alt_text"], obj["descricao_objeto"], obj["flags"] = resposta, "", []
        obj["json_valido"] = False
    print(f"[{n}/{len(objetos)}] {obj['titulo']}: {obj['alt_text'][:80]}... | flags: {len(obj['flags'])}")

## Etapa 7 — Verificação automática (com qualidade de flags)

Checagens de texto herdadas: JSON válido; povo no alt-text; artefato no alt e no nível 2; ≤30 palavras; atribuição ao registro (formulações variadas); frases-etiqueta na abertura; "foi aquisição em"; afirmações de ausência; foto vazando; especulação; frases vazias; "fundo" no alt; caso-referência do Abano.

Novidade da v5: **as flags também são verificadas** — flag de fundo liso conta como ruído (fundo de estúdio não é artefato) e flag que afirma ausência ("sem etiquetas visíveis") é defeito. Na v4, 12 das 18 flags eram ruído; a checagem agora mede isso.

In [ ]:
TERMOS_ARTEFATO = ["cartela", "paleta", "numeração", "marcação", "etiqueta", "régua", "suporte"]
ABERTURAS_ETIQUETA = ["o objeto é", "trata-se de"]  # só conta se ABRE o texto, não em qualquer posição
TERMOS_AUSENCIA = ["não há", "sem etiqueta", "sem sinais", "sem evidência", "sem artefatos", "sem marcas"]
TERMOS_FOTO = ["posicionad", "inclinad", "enquadr", "fotografia", "na imagem", "da imagem"]
TERMOS_ESPECULACAO = ["sugere", "sugerindo", "parece ", "parecendo", "possivelmente"]
FRASES_VAZIAS = ["porte médio", "uso prático", "uso frequente", "sinais de uso", "forma funcional", "forma é funcional"]

def tem_atribuicao(texto):
    t = texto.lower()
    return "registro" in t or "catálogo" in t or "catalogo" in t

problemas_totais = 0
for obj in objetos:
    p = []
    if not obj["json_valido"]:
        p.append("JSON inválido")
    povo = obj["registro"]["Povo"]
    a = obj["alt_text"].lower()
    if povo and povo.split()[0].lower() not in a:
        p.append(f"povo '{povo}' ausente do alt")
    for termo in TERMOS_ARTEFATO:
        if termo in a:
            p.append(f"artefato no alt ('{termo}')")
    if "fundo" in a:
        p.append("'fundo' no alt (fundo de estúdio? base da peça vira material)")
    if len(obj["alt_text"].split()) > 30:
        p.append(f"{len(obj['alt_text'].split())} palavras")
    d = obj["descricao_objeto"].lower().strip()
    if obj["descricao_objeto"]:
        if not tem_atribuicao(obj["descricao_objeto"]):
            p.append("nível 2 sem atribuição ao registro")
        if any(d.startswith(ab) for ab in ABERTURAS_ETIQUETA):
            p.append("nível 2 abre com frase-etiqueta")
        if "a função é" in d:
            p.append("frase-etiqueta ('a função é')")
        if "aquisição em" in d:
            p.append("'foi aquisição em' (usar 'adquirido em')")
        for termo in TERMOS_ARTEFATO:
            if termo in d:
                p.append(f"artefato no nível 2 ('{termo}')")
        for termo in TERMOS_AUSENCIA:
            if termo in d:
                p.append(f"afirmação de ausência ('{termo}')")
        for termo in TERMOS_FOTO:
            if termo in d:
                p.append(f"foto no nível 2 ('{termo}')")
    for nome, texto in [("alt", a), ("nível 2", d)]:
        for termo in TERMOS_ESPECULACAO:
            if termo in texto:
                p.append(f"especulação no {nome} ('{termo.strip()}')")
        for termo in FRASES_VAZIAS:
            if termo in texto:
                p.append(f"frase vazia no {nome} ('{termo}')")
    # qualidade das flags (aperto da v5): fundo não é artefato; flag nunca afirma ausência
    for f in obj["flags"]:
        det = f["detalhe"].lower()
        if f["tipo"] == "artefato_estudio" and "fundo" in det and not any(t in det for t in TERMOS_ARTEFATO):
            p.append("flag de fundo (ruído — fundo de estúdio não é artefato)")
        if any(t in det for t in ["sem ", "não há"]):
            p.append("flag afirmando ausência")
    obj["problemas"] = p
    problemas_totais += len(p)
    status = "✓" if not p else "⚠ " + "; ".join(p)
    print(f"{obj['id']} {obj['titulo'][:30]:30} {status}")

abano = next(o for o in objetos if o["id"] == 63283)
abano_ok = abano["alt_text"].strip().lower().startswith("detalhe") and tem_atribuicao(abano["descricao_objeto"])
print(f"\nCaso-referência Abano: {'✓ passou' if abano_ok else '✗ FALHOU'}")
print(f"Total: {sum(1 for o in objetos if not o['problemas'])}/{len(objetos)} objetos sem problemas | {sum(len(o['flags']) for o in objetos)} flags geradas")

In [ ]:
# Etapa 8 — Salvar no Drive (arquivo v5 — os resultados anteriores ficam preservados)
resultado = {
    "notebook": "04_pipeline_completo_v5",
    "modelo": MODELO,
    "embedding": "Qwen/Qwen3-Embedding-0.6B",
    "rubrica_versao": rubrica["versao"],
    "prompt_observacao": PROMPT_OBSERVACAO_V2,
    "prompt_redacao_v8": PROMPT_REDACAO_V8,
    "itens": [
        {"id": o["id"], "titulo": o["titulo"], "registro": o["registro"],
         "observacao": o["observacao"], "alt_text": o["alt_text"],
         "descricao_objeto": o["descricao_objeto"], "flags": o["flags"],
         "diretrizes_usadas": o["diretrizes_usadas"],
         "json_valido": o["json_valido"], "problemas": o["problemas"]}
        for o in objetos
    ],
}
destino = f"{PROJETO}/resultados/04_pipeline_completo_v5.json"
with open(destino, "w", encoding="utf-8") as f:
    json.dump(resultado, f, ensure_ascii=False, indent=2)
print(f"salvo no Drive ✓  {destino}")

---

## Fim — o que fazer agora

Duas coisas, nesta ordem:

1. Avise o Claude que o Notebook 04 **v5** terminou — ele analisa e compara com a v4 (a alucinação da arara sumiu? a roseta ganhou atribuição? as flags limparam?).
2. Depois, rode o **Notebook 05 (bake-off de redator)**: ele reaproveita as observações salvas por este notebook — sem reprocessar nenhuma imagem — e gera as mesmas redações com o **Gemma 3 12B**, para comparar qual modelo escreve melhor sob as mesmas regras.

**O que este notebook prova:** o pipeline com o prompt congelado (v8) após duas rodadas de revisão editorial — a versão que vai para a avaliação oficial (E8–E10). **O que ainda não prova:** as métricas nos 40 casos e o julgamento da avaliação cega.